# Notebook 02 — Dimension Tables

Build all external dimension tables saved to `data/external/`.

| Table | Source | Status |
|-------|--------|--------|
| `dim_calendar.csv` | `holidays` library + custom logic | ☐ |
| `dim_weather.csv` | Open-Meteo API | ☐ |
| `dim_events.csv` | Simulated CSV | ☐ |
| `dim_energy.csv` | GME PUN real data + 2026 scenario | ☐ |

## 1 — dim_calendar

**Source:** `holidays` library (Italy subdiv='CO' + Switzerland cantons TI/GR/ZH/BE) + custom ponte logic + meteorological seasons.

**Output:** `data/external/dim_calendar.csv`  
**Columns:** `date | is_weekend | is_holiday | is_ponte | is_swiss_holiday | season`

In [ ]:
import pandas as pd
import holidays
from datetime import timedelta
from pathlib import Path

# ── Configuration ──────────────────────────────────────────────────────────────
YEARS = [2023, 2024]
DATE_RANGE = pd.date_range(start='2023-01-01', end='2024-12-31', freq='D')
SWISS_CANTONS = ['TI', 'GR', 'ZH', 'BE']

OUTPUT_PATH = Path('../data/external/dim_calendar.csv')

# ── Italian holidays: national + Como (subdiv='CO') ───────────────────────────
it_holidays = holidays.Italy(subdiv='CO', years=YEARS)
it_holidays_set = set(it_holidays.keys())

# ── Swiss holidays: union of 4 cantons ────────────────────────────────────────
swiss_holidays_set = set()
for canton in SWISS_CANTONS:
    h = holidays.Switzerland(subdiv=canton, years=YEARS)
    swiss_holidays_set.update(h.keys())

print(f"Italian holidays (CO): {len(it_holidays_set)} dates")
print(f"Swiss holidays (TI+GR+ZH+BE): {len(swiss_holidays_set)} dates")

In [ ]:
def compute_ponte(date_range, holidays_set):
    """
    Mark bridge days (ponti) using two cases:

    Case 1 — LATERAL bridge:
        A weekday adjacent (±1 day) to BOTH a weekday holiday and a weekend day.
        Example: Thu=holiday → Fri is ponte (Thu holiday + Sat weekend on either side)

    Case 2 — CENTRAL bridge:
        A weekday flanked by weekday holidays on BOTH sides within ±2 days.
        Example: Mon=holiday + Wed=holiday → Tue between them is ponte centrale.

    Key rule: only WEEKDAY holidays trigger a ponte.
        A holiday falling on Sat/Sun already coincides with the day off —
        there is no gap to bridge, so it is excluded from ponte detection.
    """
    def is_weekday_holiday(d):
        """True only if d is a holiday AND falls on a weekday (Mon–Fri)."""
        return d in holidays_set and d.weekday() < 5

    ponte_set = set()

    for d in date_range:
        date_obj = d.date()

        # A weekend day or holiday itself cannot be a ponte
        if date_obj.weekday() >= 5 or date_obj in holidays_set:
            continue

        left1  = date_obj - timedelta(days=1)
        right1 = date_obj + timedelta(days=1)
        left2  = date_obj - timedelta(days=2)
        right2 = date_obj + timedelta(days=2)
        adj_days = [left1, right1]

        adj_holiday = any(is_weekday_holiday(n) for n in adj_days)
        adj_weekend = any(n.weekday() >= 5 for n in adj_days)

        # Case 1: lateral — weekday holiday on one side, weekend on the other
        if adj_holiday and adj_weekend:
            ponte_set.add(date_obj)
            continue

        # Case 2: central — weekday holidays on both sides within ±2 days
        left_holiday  = is_weekday_holiday(left1) or is_weekday_holiday(left2)
        right_holiday = is_weekday_holiday(right1) or is_weekday_holiday(right2)
        if left_holiday and right_holiday:
            ponte_set.add(date_obj)

    return ponte_set

ponte_set = compute_ponte(DATE_RANGE, it_holidays_set)
print(f"Ponte days detected: {len(ponte_set)}")
print("Ponte dates:")
for d in sorted(ponte_set):
    neighbor_info = []
    for delta in [-2, -1, 1, 2]:
        n = d + timedelta(days=delta)
        tag = []
        if n in it_holidays_set: tag.append('HOL')
        if n.weekday() >= 5:     tag.append('WKD')
        if tag:
            neighbor_info.append(f"{n}({'|'.join(tag)})")
    print(f"  {d} ({d.strftime('%A')}) ← neighbors: {', '.join(neighbor_info)}")

In [ ]:
def get_season(month):
    """Meteorological seasons (Northern Hemisphere)."""
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    else:
        return 'autumn'

# ── Build dim_calendar ─────────────────────────────────────────────────────────
dim_calendar = pd.DataFrame({'date': DATE_RANGE})
dim_calendar['date'] = dim_calendar['date'].dt.date

dim_calendar['is_weekend']       = dim_calendar['date'].apply(lambda d: d.weekday() >= 5)
dim_calendar['is_holiday']       = dim_calendar['date'].apply(lambda d: d in it_holidays_set)
dim_calendar['is_ponte']         = dim_calendar['date'].apply(lambda d: d in ponte_set)
dim_calendar['is_swiss_holiday'] = dim_calendar['date'].apply(lambda d: d in swiss_holidays_set)
dim_calendar['season']           = dim_calendar['date'].apply(lambda d: get_season(d.month))

# is_high_season: placeholder Jun–Sep, to be revised after join with raw_sales_pos
# Real boundaries will be derived from daily covers distribution in Notebook 03.
dim_calendar['is_high_season']   = dim_calendar['date'].apply(lambda d: d.month in [6, 7, 8, 9])

# ── Sanity checks ──────────────────────────────────────────────────────────────
print(f"Shape: {dim_calendar.shape}")
print(f"\nValue counts — is_holiday      : {dim_calendar['is_holiday'].sum()} days")
print(f"Value counts — is_ponte        : {dim_calendar['is_ponte'].sum()} days")
print(f"Value counts — is_weekend      : {dim_calendar['is_weekend'].sum()} days")
print(f"Value counts — is_swiss_holiday: {dim_calendar['is_swiss_holiday'].sum()} days")
print(f"Value counts — is_high_season  : {dim_calendar['is_high_season'].sum()} days")
print(f"\nSeason distribution:\n{dim_calendar['season'].value_counts().sort_index()}")
print("\nFirst 10 rows:")
display(dim_calendar.head(10))

In [ ]:
# ── AWAIT GIOVANNI APPROVAL before saving ─────────────────────────────────────
# Review the output above (shape, ponte dates, counts) before running this cell.

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
dim_calendar.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH.resolve()}")

## 2 — dim_weather

**Source:** Open-Meteo historical archive API — free, no API key required.  
**Coordinates:** Como (45.81°N, 9.09°E)  
**Variables:** `temperature_2m_max`, `temperature_2m_min`, `precipitation_sum`  
**Output:** `data/external/dim_weather.csv`  
**Columns:** `date | avg_temp | rain_mm | is_bad_weather`

In [ ]:
import requests
import pandas as pd
from pathlib import Path

# ── API call ───────────────────────────────────────────────────────────────────
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 45.81,
    "longitude": 9.09,
    "start_date": "2023-01-01",
    "end_date": "2024-12-31",
    "daily": ["temperature_2m_max", "temperature_2m_min", "precipitation_sum"],
    "timezone": "Europe/Rome"
}

response = requests.get(url, params=params)
response.raise_for_status()
data = response.json()

# ── Build DataFrame ────────────────────────────────────────────────────────────
daily = data["daily"]
weather_raw = pd.DataFrame({
    "date":     pd.to_datetime(daily["time"]).date,
    "temp_max": daily["temperature_2m_max"],
    "temp_min": daily["temperature_2m_min"],
    "rain_mm":  daily["precipitation_sum"],
})

weather_raw["avg_temp"] = ((weather_raw["temp_max"] + weather_raw["temp_min"]) / 2).round(1)
weather_raw = weather_raw.drop(columns=["temp_max", "temp_min"])

print(f"API returned: {len(weather_raw)} rows")
print(f"Date range  : {weather_raw['date'].min()} → {weather_raw['date'].max()}")
print(f"NaN count   :\n{weather_raw.isna().sum()}")

In [ ]:
# ── Join on dim_calendar — gap check ──────────────────────────────────────────
dim_calendar = pd.read_csv('../data/external/dim_calendar.csv', parse_dates=['date'])
dim_calendar['date'] = dim_calendar['date'].dt.date

weather_raw['date'] = pd.to_datetime(weather_raw['date']).dt.date

merged = dim_calendar[['date']].merge(weather_raw, on='date', how='left')

gaps = merged[merged['avg_temp'].isna()]

print(f"dim_calendar rows : {len(dim_calendar)}")
print(f"weather API rows  : {len(weather_raw)}")
print(f"After LEFT JOIN   : {len(merged)}")
print(f"Gap dates (NaN)   : {len(gaps)}")

if len(gaps) == 0:
    print("✓ No gaps — all dates covered.")
else:
    print("⚠ Missing dates in weather data:")
    print(gaps['date'].tolist())

In [ ]:
# ── Distribution — is_bad_weather threshold choice ────────────────────────────
print("── avg_temp distribution ──")
print(merged['avg_temp'].describe().round(1))
print(f"\nDays below 5°C  : {(merged['avg_temp'] < 5).sum()}")
print(f"Days below 0°C  : {(merged['avg_temp'] < 0).sum()}")

print("\n── rain_mm distribution ──")
print(merged['rain_mm'].describe().round(1))
print(f"\nDays with rain >  0 mm: {(merged['rain_mm'] > 0).sum()}")
print(f"Days with rain >  5 mm: {(merged['rain_mm'] > 5).sum()}")
print(f"Days with rain > 10 mm: {(merged['rain_mm'] > 10).sum()}")
print(f"Days with rain > 20 mm: {(merged['rain_mm'] > 20).sum()}")

# Threshold confirmed: rain_mm > 10 OR avg_temp < 5
# rain > 10mm = heavy rain deterrent (pre-alpine convective pattern, peaks May/Oct)
# temp < 5°C  = cold deterrent (winter complement, 46 days)

# ── Build final dim_weather ────────────────────────────────────────────────────
dim_weather = merged[['date', 'avg_temp', 'rain_mm']].copy()
dim_weather['is_bad_weather'] = (dim_weather['rain_mm'] > 10) | (dim_weather['avg_temp'] < 5)

print(f"\nis_bad_weather days: {dim_weather['is_bad_weather'].sum()} ({dim_weather['is_bad_weather'].mean()*100:.1f}%)")
print(f"\nFirst 10 rows:")
display(dim_weather.head(10))

In [ ]:
OUTPUT_WEATHER = Path('../data/external/dim_weather.csv')
dim_weather.to_csv(OUTPUT_WEATHER, index=False)
print(f"Saved: {OUTPUT_WEATHER.resolve()}")
print(f"Shape: {dim_weather.shape}")

## 3 — dim_events

**Source:** Real 2023-2024 events scraped from Wikipedia, official club/venue sites + manual entries.  
**Note on quality:** dates marked `# EST` are estimated from historical patterns — verify and correct if needed.  
**Output:** `data/external/dim_events.csv`  
**Columns:** `date | city | event_name | event_type | magnitude | radius_km`

In [ ]:
import pandas as pd
from pathlib import Path

def days(start, end):
    """Return list of date strings for a range inclusive."""
    return [str(d.date()) for d in pd.date_range(start=start, end=end, freq='D')]

def single(date):
    return [date]

events = []

def add(date_list, city, name, etype, magnitude, radius_km):
    for d in date_list:
        events.append({
            "date": d, "city": city, "event_name": name,
            "event_type": etype, "magnitude": magnitude, "radius_km": radius_km
        })

# ══════════════════════════════════════════════════════════════════════════════
# COMO — LOCAL & PROVINCIAL EVENTS
# ══════════════════════════════════════════════════════════════════════════════

# ── Como 1907 — Serie B home matches 2023-24 (source: Wikipedia) ──────────────
serie_b = [
    ("2023-08-26", "vs Reggiana"),    ("2023-09-17", "vs Ternana"),
    ("2023-09-27", "vs Sampdoria"),   ("2023-10-28", "vs Catanzaro"),
    ("2023-11-25", "vs Feralpisalò"), ("2023-11-28", "vs Lecco"),
    ("2023-12-10", "vs Modena"),      ("2023-12-23", "vs Palermo"),
    ("2024-01-13", "vs Spezia"),      ("2024-01-27", "vs Ascoli"),
    ("2024-02-09", "vs Brescia"),     ("2024-02-24", "vs Parma"),
    ("2024-03-03", "vs Venezia"),     ("2024-03-16", "vs Pisa"),
    ("2024-04-01", "vs Südtirol"),    ("2024-04-13", "vs Bari"),
    ("2024-05-01", "vs Cittadella"),  ("2024-05-10", "vs Cosenza"),
]
for date, opp in serie_b:
    add(single(date), "Como", f"Como 1907 {opp} (Serie B)",
        "sports", magnitude=1, radius_km=2)

# ── Como 1907 — Serie A home matches 2024-25 through Dec 2024 (source: Wikipedia) ──
serie_a = [
    ("2024-09-14", "vs Bologna"),      ("2024-09-29", "vs Hellas Verona"),
    ("2024-10-19", "vs Parma"),        ("2024-10-31", "vs Lazio"),
    ("2024-11-24", "vs Fiorentina"),   ("2024-11-30", "vs Monza"),
]
for date, opp in serie_a:
    add(single(date), "Como", f"Como 1907 {opp} (Serie A)",
        "sports", magnitude=2, radius_km=2)

# ── Fiera di Sant'Abbondio (source: confirmed by user — 31 agosto ogni anno) ──
add(single("2023-08-31"), "Como", "Fiera di Sant'Abbondio", "festival", 2, 3)
add(single("2024-08-31"), "Como", "Fiera di Sant'Abbondio", "festival", 2, 3)

# ── Palio del Baradello — EST: last weekend of September ─────────────────────
add(days("2023-09-23", "2023-09-24"), "Como", "Palio del Baradello", "festival", 2, 3)  # EST
add(days("2024-09-21", "2024-09-22"), "Como", "Palio del Baradello", "festival", 2, 3)  # EST

# ── Mercatini di Natale Como (December 1-26 both years) ──────────────────────
add(days("2023-12-01", "2023-12-26"), "Como", "Mercatini di Natale Como", "market", 2, 3)
add(days("2024-12-01", "2024-12-26"), "Como", "Mercatini di Natale Como", "market", 2, 3)

# ── Teatro Sociale Como — major shows only (source: teatrosocialecomo.it) ─────
teatro = [
    ("2023-09-28", "2023-09-30", "Die Zauberflöte (Mozart) — Teatro Sociale"),
    ("2023-10-03", "2023-10-03", "Carmina Burana — Teatro Sociale"),
    ("2023-11-08", "2023-11-09", "La vita davanti a sé — Teatro Sociale"),
    ("2023-12-08", "2023-12-10", "Don Carlo (Verdi) — Teatro Sociale"),
    ("2024-03-24", "2024-03-24", "Sister Act (musical) — Teatro Sociale"),
    ("2024-04-10", "2024-04-11", "Grease (musical) — Teatro Sociale"),
]
for start, end, name in teatro:
    add(days(start, end), "Como", name, "concert", 1, 2)

# ══════════════════════════════════════════════════════════════════════════════
# CERNOBBIO — VILLA D'ESTE (radius ~7km from Como centre)
# ══════════════════════════════════════════════════════════════════════════════

# ── Concorso d'Eleganza Villa d'Este (source: Wikipedia) ─────────────────────
add(days("2023-05-19", "2023-05-21"), "Cernobbio", "Concorso d'Eleganza Villa d'Este", "festival", 3, 7)
add(days("2024-05-24", "2024-05-26"), "Cernobbio", "Concorso d'Eleganza Villa d'Este", "festival", 3, 7)

# ══════════════════════════════════════════════════════════════════════════════
# COMO AREA — TOURISM PEAKS (radius 0 = applies to entire lake area)
# ══════════════════════════════════════════════════════════════════════════════

# ── Ferragosto tourism peak (Aug 10-18 both years) ───────────────────────────
add(days("2023-08-10", "2023-08-18"), "Como", "Ferragosto — Tourism Peak", "tourism_peak", 3, 0)
add(days("2024-08-10", "2024-08-18"), "Como", "Ferragosto — Tourism Peak", "tourism_peak", 3, 0)

# ── Pasqua + Pasquetta (Easter 2023: Apr 9-10 | Easter 2024: Mar 31 - Apr 1) ─
add(days("2023-04-08", "2023-04-10"), "Como", "Pasqua e Pasquetta", "tourism_peak", 2, 0)
add(days("2024-03-30", "2024-04-01"), "Como", "Pasqua e Pasquetta", "tourism_peak", 2, 0)

# ── Ponte Capodanno (Dec 30 - Jan 2, only dates within 2023-2024 range) ──────
add(days("2023-01-01", "2023-01-02"), "Como", "Ponte Capodanno", "tourism_peak", 2, 0)
add(days("2023-12-30", "2024-01-02"), "Como", "Ponte Capodanno", "tourism_peak", 2, 0)

# ══════════════════════════════════════════════════════════════════════════════
# MONZA (radius ~45km from Como centre)
# ══════════════════════════════════════════════════════════════════════════════

# ── GP Italia F1 Monza (source: Wikipedia) ────────────────────────────────────
add(days("2023-09-01", "2023-09-03"), "Monza", "GP Italia F1", "sports", 3, 45)
add(days("2024-08-30", "2024-09-01"), "Monza", "GP Italia F1", "sports", 3, 45)

# ══════════════════════════════════════════════════════════════════════════════
# MILANO (radius ~50km from Como centre)
# ══════════════════════════════════════════════════════════════════════════════

# ── Salone del Mobile (source: Wikipedia) ─────────────────────────────────────
add(days("2023-04-18", "2023-04-23"), "Milano", "Salone del Mobile 2023", "fair", 3, 50)
add(days("2024-04-16", "2024-04-21"), "Milano", "Salone del Mobile 2024", "fair", 3, 50)

# ── EICMA (source: Wikipedia 2023 confirmed | 2024 EST pattern) ──────────────
add(days("2023-11-07", "2023-11-12"), "Milano", "EICMA 2023", "fair", 3, 50)
add(days("2024-11-05", "2024-11-10"), "Milano", "EICMA 2024", "fair", 3, 50)  # EST

# ── Milano Fashion Week (EST: typical Feb/Sep dates ±1-2 days) ───────────────
add(days("2023-02-21", "2023-02-27"), "Milano", "Milano Fashion Week Feb 2023", "fair", 3, 50)  # EST
add(days("2023-09-19", "2023-09-25"), "Milano", "Milano Fashion Week Sep 2023", "fair", 3, 50)  # EST
add(days("2024-02-20", "2024-02-26"), "Milano", "Milano Fashion Week Feb 2024", "fair", 3, 50)  # EST
add(days("2024-09-17", "2024-09-23"), "Milano", "Milano Fashion Week Sep 2024", "fair", 3, 50)  # EST

# ══════════════════════════════════════════════════════════════════════════════
# BUILD DATAFRAME
# ══════════════════════════════════════════════════════════════════════════════
dim_events = pd.DataFrame(events)
dim_events['date'] = pd.to_datetime(dim_events['date']).dt.date

# Keep only dates within our 2023-2024 range
start_date = pd.Timestamp("2023-01-01").date()
end_date   = pd.Timestamp("2024-12-31").date()
dim_events = dim_events[(dim_events['date'] >= start_date) & (dim_events['date'] <= end_date)]
dim_events = dim_events.sort_values('date').reset_index(drop=True)

print(f"Shape: {dim_events.shape}")
print(f"Date range: {dim_events['date'].min()} → {dim_events['date'].max()}")
print(f"\nevent_type counts:\n{dim_events['event_type'].value_counts()}")
print(f"\ncity counts:\n{dim_events['city'].value_counts()}")
print(f"\nmagnitude counts:\n{dim_events['magnitude'].value_counts().sort_index()}")
print(f"\nUnique event days (distinct dates with ≥1 event): {dim_events['date'].nunique()}")
print("\nSample — first 15 rows:")
display(dim_events.head(15))

In [ ]:
OUTPUT_EVENTS = Path('../data/external/dim_events.csv')
dim_events.to_csv(OUTPUT_EVENTS, index=False)
print(f"Saved: {OUTPUT_EVENTS.resolve()}")
print(f"Shape: {dim_events.shape}")